# DR-Detect: Comprehensive Results Visualization Report

**Report Generated**: 2026-04-03  
**Purpose**: Visualize all training and evaluation results from the outputs directory

---

## Table of Contents
1. [Setup & Data Loading](#setup)
2. [Training Results](#training)
3. [Model Performance Comparison](#comparison)
4. [External Evaluation Results](#external)
5. [Calibration Analysis](#calibration)
6. [Uncertainty Analysis](#uncertainty)
7. [Grad-CAM Visualization](#gradcam)
8. [Summary Tables](#summary)

---
<a id="setup"></a>
## 1. Setup & Data Loading

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display, Markdown, Image
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Define paths
BASE_DIR = Path('../outputs')
RESULTS_DIR = BASE_DIR / 'results'
LOGS_DIR = BASE_DIR / 'logs'
FIGURES_DIR = BASE_DIR / 'figures'
CHECKPOINTS_DIR = BASE_DIR / 'checkpoints'

print("✓ Imports successful")
print(f"✓ Base directory: {BASE_DIR.resolve()}")
print(f"✓ Results directory: {RESULTS_DIR.resolve()}")

In [ ]:
# Helper functions
def load_json(filepath):
    """Load JSON file and return dictionary"""
    with open(filepath, 'r') as f:
        return json.load(f)

def display_json_as_table(data, title=""):
    """Display nested JSON as formatted table"""
    if title:
        display(Markdown(f"### {title}"))
    
    if isinstance(data, dict):
        df = pd.DataFrame([data]).T
        df.columns = ['Value']
        display(df)
    else:
        display(pd.DataFrame(data))

def plot_training_curves(history_data, title="Training Curves"):
    """Plot training and validation metrics over epochs"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    
    epochs = range(1, len(history_data['train_loss']) + 1)
    
    # Loss
    axes[0, 0].plot(epochs, history_data['train_loss'], 'b-', label='Train Loss', linewidth=2)
    axes[0, 0].plot(epochs, history_data['val_loss'], 'r-', label='Val Loss', linewidth=2)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Loss Curves')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[0, 1].plot(epochs, history_data['train_acc'], 'b-', label='Train Acc', linewidth=2)
    axes[0, 1].plot(epochs, history_data['val_acc'], 'r-', label='Val Acc', linewidth=2)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].set_title('Accuracy Curves')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Kappa
    axes[1, 0].plot(epochs, history_data['val_kappa'], 'g-', label='Val Kappa', linewidth=2)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Quadratic Kappa')
    axes[1, 0].set_title('Validation Kappa')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # AUC
    axes[1, 1].plot(epochs, history_data['val_auc'], 'purple', label='Val AUC', linewidth=2)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('AUC')
    axes[1, 1].set_title('Validation AUC (Referable DR)')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

print("✓ Helper functions loaded")

---
<a id="training"></a>
## 2. Training Results

### 2.1 List All Training Runs

In [ ]:
# Discover all training result files
training_files = list(RESULTS_DIR.glob('*fold*_metrics.json'))
history_files = list(LOGS_DIR.glob('*history.json'))

print(f"Found {len(training_files)} training metric files")
print(f"Found {len(history_files)} history files\n")

# Create inventory table
inventory_data = []
for f in training_files:
    data = load_json(f)
    if 'run_info' in data:
        inventory_data.append({
            'Model': data['run_info'].get('model_name', 'Unknown'),
            'Fold': data['run_info'].get('fold', 'N/A'),
            'Timestamp': data['run_info'].get('timestamp', 'N/A'),
            'Runtime': data['run_info'].get('runtime_formatted', 'N/A'),
            'Best Epoch': data['best_metrics'].get('epoch', 'N/A'),
            'Best Val Kappa': f"{data['best_metrics'].get('val_kappa', 0):.4f}",
            'Best Val Acc': f"{data['best_metrics'].get('val_acc', 0):.4f}"
        })

if inventory_data:
    df_inventory = pd.DataFrame(inventory_data)
    display(Markdown("### Training Runs Inventory"))
    display(df_inventory)

### 2.2 Baseline ResNet-50 Training (Original)

In [ ]:
# Load original baseline training history
baseline_history_path = LOGS_DIR / 'baseline_resnet50_fold0_history.json'
if baseline_history_path.exists():
    baseline_history = load_json(baseline_history_path)
    
    # Plot training curves
    fig = plot_training_curves(baseline_history, "Baseline ResNet-50 Training (Original)")
    plt.show()
    
    # Show metrics
    baseline_metrics_path = RESULTS_DIR / 'baseline_resnet50_fold0_metrics.json'
    if baseline_metrics_path.exists():
        baseline_metrics = load_json(baseline_metrics_path)
        display_json_as_table(baseline_metrics, "Baseline Training Metrics")
else:
    print("❌ Baseline history file not found")

### 2.3 Baseline ResNet-50 Retrained (2026-04-03, data_split)

In [ ]:
# Load retrained baseline metrics
retrained_metrics_path = RESULTS_DIR / 'baseline_resnet50_20260403_120248_fold0_metrics.json'
if retrained_metrics_path.exists():
    retrained_metrics = load_json(retrained_metrics_path)
    
    display(Markdown("### Retrained Baseline (data_split) - Run Info"))
    display_json_as_table(retrained_metrics['run_info'])
    
    display(Markdown("### Best Validation Metrics (Epoch 15)"))
    display_json_as_table(retrained_metrics['best_metrics'])
    
    display(Markdown("### Final Metrics (Epoch 20)"))
    display_json_as_table(retrained_metrics['final_metrics'])
    
    display(Markdown("### Hyperparameters"))
    display_json_as_table(retrained_metrics['hyperparameters'])
    
    # Per-class metrics
    display(Markdown("### Per-Class Performance (Best Epoch)"))
    per_class_data = {
        'Grade': ['0-No DR', '1-Mild', '2-Moderate', '3-Severe', '4-Proliferative'],
        'Precision': retrained_metrics['best_metrics']['val_precision_per_class'],
        'Recall': retrained_metrics['best_metrics']['val_recall_per_class'],
        'F1': retrained_metrics['best_metrics']['val_f1_per_class']
    }
    df_per_class = pd.DataFrame(per_class_data)
    display(df_per_class.style.format({
        'Precision': '{:.4f}',
        'Recall': '{:.4f}',
        'F1': '{:.4f}'
    }))
else:
    print("❌ Retrained baseline metrics not found")

### 2.4 CBAM-ResNet50 Training

In [ ]:
# Load CBAM training history
cbam_history_path = LOGS_DIR / 'cbam_resnet50_20260331_114901_fold0_history.json'
if cbam_history_path.exists():
    cbam_history = load_json(cbam_history_path)
    
    # Plot training curves
    fig = plot_training_curves(cbam_history, "CBAM-ResNet50 Training")
    plt.show()
    
    # Show metrics
    cbam_metrics_path = RESULTS_DIR / 'cbam_resnet50_20260331_114901_fold0_metrics.json'
    if cbam_metrics_path.exists():
        cbam_metrics = load_json(cbam_metrics_path)
        display_json_as_table(cbam_metrics['best_metrics'], "CBAM Best Metrics")
else:
    print("❌ CBAM history file not found")

### 2.5 Display Existing Training Figures

In [ ]:
# Display training-related figures
training_figure_files = [
    'baseline_resnet50_training_curves.png',
    'baseline_resnet50_confusion_matrix.png',
    'baseline_resnet50_per_class_recall.png'
]

for fig_name in training_figure_files:
    fig_path = FIGURES_DIR / fig_name
    if fig_path.exists():
        display(Markdown(f"#### {fig_name.replace('_', ' ').replace('.png', '').title()}"))
        display(Image(filename=str(fig_path)))
    else:
        print(f"⚠️  {fig_name} not found")

---
<a id="comparison"></a>
## 3. Model Performance Comparison

### 3.1 Internal Validation Comparison

In [ ]:
# Compare models on internal validation
comparison_data = []

# Original Baseline
if baseline_metrics_path.exists():
    bm = load_json(baseline_metrics_path)
    if isinstance(bm, dict) and 'val_acc' in bm:
        comparison_data.append({
            'Model': 'Baseline (Original)',
            'Val Accuracy': f"{bm.get('val_acc', 0):.4f}",
            'Val Kappa': f"{bm.get('val_kappa', 0):.4f}",
            'Val AUC': f"{bm.get('val_auc', 0):.4f}"
        })

# Retrained Baseline
if retrained_metrics_path.exists():
    rm = load_json(retrained_metrics_path)
    comparison_data.append({
        'Model': 'Baseline (Retrained)',
        'Val Accuracy': f"{rm['best_metrics']['val_acc']:.4f}",
        'Val Kappa': f"{rm['best_metrics']['val_kappa']:.4f}",
        'Val AUC': f"{rm['best_metrics']['val_auc']:.4f}"
    })

# CBAM
cbam_metrics_path = RESULTS_DIR / 'cbam_resnet50_20260331_114901_fold0_metrics.json'
if cbam_metrics_path.exists():
    cm = load_json(cbam_metrics_path)
    comparison_data.append({
        'Model': 'CBAM-ResNet50',
        'Val Accuracy': f"{cm['best_metrics']['val_acc']:.4f}",
        'Val Kappa': f"{cm['best_metrics']['val_kappa']:.4f}",
        'Val AUC': f"{cm['best_metrics']['val_auc']:.4f}"
    })

if comparison_data:
    df_comparison = pd.DataFrame(comparison_data)
    display(Markdown("### Internal Validation Performance"))
    display(df_comparison)
    
    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    metrics = ['Val Accuracy', 'Val Kappa', 'Val AUC']
    
    for i, metric in enumerate(metrics):
        values = [float(x) for x in df_comparison[metric]]
        axes[i].bar(df_comparison['Model'], values, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
        axes[i].set_ylabel(metric)
        axes[i].set_title(f'{metric} Comparison')
        axes[i].set_ylim([min(values) - 0.05, 1.0])
        axes[i].grid(True, alpha=0.3, axis='y')
        
        # Add value labels
        for j, v in enumerate(values):
            axes[i].text(j, v + 0.01, f'{v:.4f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

---
<a id="external"></a>
## 4. External Evaluation Results

### 4.1 APTOS Test Split (Retrained Baseline)

In [ ]:
# Load APTOS test results
aptos_test_path = RESULTS_DIR / 'baseline_aptos_test_20260403_124108_20T_550img_metrics.json'
if aptos_test_path.exists():
    aptos_test = load_json(aptos_test_path)
    
    display(Markdown("### APTOS Test Set Results (N=550)"))
    
    # Main metrics
    main_metrics = {
        'Accuracy': aptos_test['accuracy'],
        'Quadratic Kappa': aptos_test['quadratic_kappa'],
        'Binary Referable AUC': aptos_test['binary_referable_auc'],
        'Referable Sensitivity': aptos_test['binary_referable_sens'],
        'Referable Specificity': aptos_test['binary_referable_spec'],
        'ECE': aptos_test['ece'],
        'Brier Score': aptos_test['brier_score']
    }
    display_json_as_table(main_metrics)
    
    # Per-class metrics
    display(Markdown("### Per-Class Performance"))
    class_report = aptos_test['classification_report']
    class_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']
    
    class_data = []
    for name in class_names:
        if name in class_report:
            class_data.append({
                'Grade': name,
                'Precision': class_report[name]['precision'],
                'Recall': class_report[name]['recall'],
                'F1-Score': class_report[name]['f1-score'],
                'Support': int(class_report[name]['support'])
            })
    
    df_aptos_class = pd.DataFrame(class_data)
    display(df_aptos_class.style.format({
        'Precision': '{:.4f}',
        'Recall': '{:.4f}',
        'F1-Score': '{:.4f}'
    }))
    
    # Display figures
    aptos_figures = [
        'baseline_aptos_test_20260403_124108_20T_550img_reliability.png',
        'baseline_aptos_test_20260403_124108_20T_550img_referral_curve.png',
        'baseline_aptos_test_20260403_124108_20T_550img_entropy_hist.png',
        'baseline_aptos_test_20260403_124108_20T_550img_conf_vs_ent.png'
    ]
    
    for fig_name in aptos_figures:
        fig_path = FIGURES_DIR / fig_name
        if fig_path.exists():
            display(Markdown(f"#### {fig_name.split('_')[-1].replace('.png', '').title()}"))
            display(Image(filename=str(fig_path), width=600))
else:
    print("❌ APTOS test results not found")

### 4.2 Messidor-2 External Test (All Models)

In [ ]:
# Load all Messidor-2 evaluation results
messidor_files = list(RESULTS_DIR.glob('*messidor2*1744img_metrics.json'))

messidor_results = []
for mf in messidor_files:
    data = load_json(mf)
    model_name = mf.stem.split('_')[0].upper()
    timestamp = data['run_info'].get('timestamp', 'Unknown')
    
    messidor_results.append({
        'Model': model_name,
        'Timestamp': timestamp,
        'Accuracy': f"{data['accuracy']:.4f}",
        'Kappa': f"{data['quadratic_kappa']:.4f}",
        'AUC': f"{data['binary_referable_auc']:.4f}",
        'Sensitivity': f"{data['binary_referable_sens']:.4f}",
        'Specificity': f"{data['binary_referable_spec']:.4f}",
        'ECE': f"{data['ece']:.4f}",
        'Brier': f"{data['brier_score']:.4f}"
    })

if messidor_results:
    df_messidor = pd.DataFrame(messidor_results)
    display(Markdown("### Messidor-2 Results (N=1744) - All Models"))
    display(df_messidor)
    print(f"\nTotal runs: {len(messidor_results)}")

### 4.3 Messidor-2 Detailed Results (Retrained Baseline)

In [ ]:
# Load detailed results for retrained baseline on Messidor-2
messidor_retrained_path = RESULTS_DIR / 'baseline_messidor2_20260403_124305_20T_1744img_metrics.json'
if messidor_retrained_path.exists():
    messidor_retrained = load_json(messidor_retrained_path)
    
    display(Markdown("### Retrained Baseline on Messidor-2 - Per-Class Performance"))
    
    class_report = messidor_retrained['classification_report']
    class_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']
    
    class_data = []
    for name in class_names:
        if name in class_report:
            class_data.append({
                'Grade': name,
                'Precision': class_report[name]['precision'],
                'Recall': class_report[name]['recall'],
                'F1-Score': class_report[name]['f1-score'],
                'Support': int(class_report[name]['support'])
            })
    
    df_messidor_class = pd.DataFrame(class_data)
    display(df_messidor_class.style.format({
        'Precision': '{:.4f}',
        'Recall': '{:.4f}',
        'F1-Score': '{:.4f}'
    }).background_gradient(subset=['Recall'], cmap='RdYlGn', vmin=0, vmax=1))
    
    # Display figures
    messidor_figures = [
        'baseline_messidor2_20260403_124305_20T_1744img_reliability.png',
        'baseline_messidor2_20260403_124305_20T_1744img_referral_curve.png',
        'baseline_messidor2_20260403_124305_20T_1744img_entropy_hist.png',
        'baseline_messidor2_20260403_124305_20T_1744img_conf_vs_ent.png'
    ]
    
    for fig_name in messidor_figures:
        fig_path = FIGURES_DIR / fig_name
        if fig_path.exists():
            display(Markdown(f"#### {fig_name.split('_')[-1].replace('.png', '').title()}"))
            display(Image(filename=str(fig_path), width=600))
else:
    print("❌ Retrained baseline Messidor-2 results not found")

### 4.4 Domain Shift Analysis

In [ ]:
# Compare performance: Internal Val → APTOS Test → Messidor-2
if retrained_metrics_path.exists() and aptos_test_path.exists() and messidor_retrained_path.exists():
    
    val_data = load_json(retrained_metrics_path)['best_metrics']
    aptos_data = load_json(aptos_test_path)
    messidor_data = load_json(messidor_retrained_path)
    
    domain_shift = {
        'Dataset': ['Internal Val', 'APTOS Test', 'Messidor-2'],
        'N Images': [623, 550, 1744],
        'Accuracy': [val_data['val_acc'], aptos_data['accuracy'], messidor_data['accuracy']],
        'Kappa': [val_data['val_kappa'], aptos_data['quadratic_kappa'], messidor_data['quadratic_kappa']],
        'AUC': [val_data['val_auc'], aptos_data['binary_referable_auc'], messidor_data['binary_referable_auc']],
        'ECE': [None, aptos_data['ece'], messidor_data['ece']]
    }
    
    df_domain = pd.DataFrame(domain_shift)
    display(Markdown("### Domain Shift Analysis (Retrained Baseline)"))
    display(df_domain.style.format({
        'Accuracy': '{:.4f}',
        'Kappa': '{:.4f}',
        'AUC': '{:.4f}',
        'ECE': lambda x: f'{x:.4f}' if x is not None else 'N/A'
    }))
    
    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    datasets = df_domain['Dataset']
    colors = ['#2ecc71', '#3498db', '#e74c3c']
    
    # Accuracy
    axes[0].plot(datasets, df_domain['Accuracy'], 'o-', linewidth=2, markersize=10, color='steelblue')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title('Accuracy Across Datasets')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim([0.5, 1.0])
    
    # Kappa
    axes[1].plot(datasets, df_domain['Kappa'], 'o-', linewidth=2, markersize=10, color='coral')
    axes[1].set_ylabel('Quadratic Kappa')
    axes[1].set_title('Kappa Across Datasets')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim([0.5, 1.0])
    
    # AUC
    axes[2].plot(datasets, df_domain['AUC'], 'o-', linewidth=2, markersize=10, color='forestgreen')
    axes[2].set_ylabel('AUC')
    axes[2].set_title('AUC Across Datasets')
    axes[2].grid(True, alpha=0.3)
    axes[2].set_ylim([0.8, 1.0])
    
    plt.tight_layout()
    plt.show()
    
    # Calculate drops
    display(Markdown("### Performance Degradation"))
    drops = {
        'Metric': ['Accuracy', 'Kappa', 'AUC'],
        'Val → APTOS Test': [
            f"{(df_domain['Accuracy'][1] - df_domain['Accuracy'][0]):.4f}",
            f"{(df_domain['Kappa'][1] - df_domain['Kappa'][0]):.4f}",
            f"{(df_domain['AUC'][1] - df_domain['AUC'][0]):.4f}"
        ],
        'Val → Messidor-2': [
            f"{(df_domain['Accuracy'][2] - df_domain['Accuracy'][0]):.4f}",
            f"{(df_domain['Kappa'][2] - df_domain['Kappa'][0]):.4f}",
            f"{(df_domain['AUC'][2] - df_domain['AUC'][0]):.4f}"
        ]
    }
    display(pd.DataFrame(drops))

---
<a id="calibration"></a>
## 5. Calibration Analysis

In [ ]:
# Display calibration-related figures
calibration_figures = list(FIGURES_DIR.glob('*reliability*.png'))

display(Markdown("### Reliability Diagrams (Calibration Curves)"))
display(Markdown("These diagrams show how well-calibrated the model predictions are. "
                 "Points close to the diagonal line indicate good calibration."))

for fig_path in sorted(calibration_figures):
    model_name = 'CBAM' if 'cbam' in fig_path.name else 'Baseline'
    dataset = 'APTOS Test' if 'aptos' in fig_path.name else 'Messidor-2'
    display(Markdown(f"#### {model_name} - {dataset}"))
    display(Image(filename=str(fig_path), width=500))

# Temperature scaling
temp_scaling_path = FIGURES_DIR / 'baseline_temp_scaling_reliability.png'
if temp_scaling_path.exists():
    display(Markdown("#### Temperature Scaling (Baseline)"))
    display(Image(filename=str(temp_scaling_path), width=500))

---
<a id="uncertainty"></a>
## 6. Uncertainty Analysis

In [ ]:
# Load uncertainty data
display(Markdown("### Entropy Histograms"))
display(Markdown("Higher entropy indicates higher model uncertainty."))

entropy_figures = list(FIGURES_DIR.glob('*entropy_hist*.png'))
for fig_path in sorted(entropy_figures)[-6:]:  # Show last 6
    if '1744img' in fig_path.name or '550img' in fig_path.name:
        display(Markdown(f"#### {fig_path.stem}"))
        display(Image(filename=str(fig_path), width=500))

In [ ]:
# Confidence vs Entropy scatter plots
display(Markdown("### Confidence vs Entropy Analysis"))
display(Markdown("These scatter plots show the relationship between prediction confidence and uncertainty."))

conf_ent_figures = list(FIGURES_DIR.glob('*conf_vs_ent*.png'))
for fig_path in sorted(conf_ent_figures)[-4:]:  # Show last 4
    if '1744img' in fig_path.name or '550img' in fig_path.name:
        display(Markdown(f"#### {fig_path.stem}"))
        display(Image(filename=str(fig_path), width=600))

In [ ]:
# Referral curves
display(Markdown("### Referral Curves (Uncertainty-Based Rejection)"))
display(Markdown("These curves show how accuracy improves when rejecting uncertain predictions."))

referral_figures = list(FIGURES_DIR.glob('*referral_curve*.png'))
for fig_path in sorted(referral_figures)[-4:]:
    if '1744img' in fig_path.name or '550img' in fig_path.name:
        display(Markdown(f"#### {fig_path.stem}"))
        display(Image(filename=str(fig_path), width=500))

In [ ]:
# Load and analyze uncertainty CSV data
uncertainty_csv_files = list(RESULTS_DIR.glob('*uncertainty.csv'))

if uncertainty_csv_files:
    display(Markdown("### Uncertainty Statistics Summary"))
    
    # Load retrained baseline APTOS test uncertainty
    aptos_unc_path = RESULTS_DIR / 'baseline_aptos_test_20260403_124108_20T_550img_uncertainty.csv'
    if aptos_unc_path.exists():
        df_aptos_unc = pd.read_csv(aptos_unc_path)
        display(Markdown("#### APTOS Test Set Uncertainty"))
        print(f"Total images: {len(df_aptos_unc)}")
        print(f"Mean entropy: {df_aptos_unc['entropy'].mean():.4f}")
        print(f"Mean confidence: {df_aptos_unc['confidence'].mean():.4f}")
        print(f"Correct predictions: {df_aptos_unc['correct'].sum()} ({df_aptos_unc['correct'].mean()*100:.2f}%)")
        
        # Show sample
        display(df_aptos_unc.head(10))
    
    # Load retrained baseline Messidor-2 uncertainty
    messidor_unc_path = RESULTS_DIR / 'baseline_messidor2_20260403_124305_20T_1744img_uncertainty.csv'
    if messidor_unc_path.exists():
        df_messidor_unc = pd.read_csv(messidor_unc_path)
        display(Markdown("#### Messidor-2 Uncertainty"))
        print(f"Total images: {len(df_messidor_unc)}")
        print(f"Mean entropy: {df_messidor_unc['entropy'].mean():.4f}")
        print(f"Mean confidence: {df_messidor_unc['confidence'].mean():.4f}")
        print(f"Correct predictions: {df_messidor_unc['correct'].sum()} ({df_messidor_unc['correct'].mean()*100:.2f}%)")
        
        # Show sample
        display(df_messidor_unc.head(10))

---
<a id="gradcam"></a>
## 7. Grad-CAM Visualization

In [ ]:
# List all Grad-CAM images
gradcam_dir = FIGURES_DIR / 'gradcam'
if gradcam_dir.exists():
    gradcam_images = list(gradcam_dir.glob('*.png'))
    gradcam_csvs = list(gradcam_dir.glob('*.csv'))
    
    display(Markdown(f"### Grad-CAM Visualizations (Total: {len(gradcam_images)} images)"))
    display(Markdown("Grad-CAM shows which regions of the image the model focuses on for its predictions."))
    
    # Load and display summary CSVs
    for csv_path in gradcam_csvs:
        if csv_path.stat().st_size > 10:  # Skip empty files
            display(Markdown(f"#### {csv_path.stem}"))
            df_gradcam = pd.read_csv(csv_path)
            if not df_gradcam.empty:
                display(df_gradcam)
    
    # Display sample Grad-CAM images
    display(Markdown("#### Sample Grad-CAM Visualizations"))
    sample_images = sorted(gradcam_images)[:12]  # Show first 12
    
    for img_path in sample_images:
        display(Markdown(f"**{img_path.name}**"))
        display(Image(filename=str(img_path), width=700))
else:
    print("❌ Grad-CAM directory not found")

---
<a id="summary"></a>
## 8. Summary Tables

### 8.1 Complete Results Summary

In [ ]:
# Create comprehensive summary table
summary_rows = []

# Retrained Baseline - Internal Val
if retrained_metrics_path.exists():
    rm = load_json(retrained_metrics_path)
    summary_rows.append({
        'Model': 'Baseline (Retrained)',
        'Dataset': 'APTOS Val',
        'N': 623,
        'Accuracy': rm['best_metrics']['val_acc'],
        'Kappa': rm['best_metrics']['val_kappa'],
        'AUC': rm['best_metrics']['val_auc'],
        'Sensitivity': rm['best_metrics']['val_sens'],
        'Specificity': rm['best_metrics']['val_spec'],
        'ECE': None,
        'Brier': None
    })

# Retrained Baseline - APTOS Test
if aptos_test_path.exists():
    at = load_json(aptos_test_path)
    summary_rows.append({
        'Model': 'Baseline (Retrained)',
        'Dataset': 'APTOS Test',
        'N': 550,
        'Accuracy': at['accuracy'],
        'Kappa': at['quadratic_kappa'],
        'AUC': at['binary_referable_auc'],
        'Sensitivity': at['binary_referable_sens'],
        'Specificity': at['binary_referable_spec'],
        'ECE': at['ece'],
        'Brier': at['brier_score']
    })

# Retrained Baseline - Messidor-2
if messidor_retrained_path.exists():
    mr = load_json(messidor_retrained_path)
    summary_rows.append({
        'Model': 'Baseline (Retrained)',
        'Dataset': 'Messidor-2',
        'N': 1744,
        'Accuracy': mr['accuracy'],
        'Kappa': mr['quadratic_kappa'],
        'AUC': mr['binary_referable_auc'],
        'Sensitivity': mr['binary_referable_sens'],
        'Specificity': mr['binary_referable_spec'],
        'ECE': mr['ece'],
        'Brier': mr['brier_score']
    })

# Add other models if available
for mf in messidor_files:
    if '20260331' in mf.name and '1744img' in mf.name:
        data = load_json(mf)
        model_name = 'CBAM' if 'cbam' in mf.name else 'Baseline (Original)'
        summary_rows.append({
            'Model': model_name,
            'Dataset': 'Messidor-2',
            'N': 1744,
            'Accuracy': data['accuracy'],
            'Kappa': data['quadratic_kappa'],
            'AUC': data['binary_referable_auc'],
            'Sensitivity': data['binary_referable_sens'],
            'Specificity': data['binary_referable_spec'],
            'ECE': data['ece'],
            'Brier': data['brier_score']
        })

if summary_rows:
    df_summary = pd.DataFrame(summary_rows)
    display(Markdown("### Complete Performance Summary"))
    display(df_summary.style.format({
        'Accuracy': '{:.4f}',
        'Kappa': '{:.4f}',
        'AUC': '{:.4f}',
        'Sensitivity': '{:.4f}',
        'Specificity': '{:.4f}',
        'ECE': lambda x: f'{x:.4f}' if x is not None else 'N/A',
        'Brier': lambda x: f'{x:.4f}' if x is not None else 'N/A'
    }).background_gradient(subset=['Kappa', 'AUC'], cmap='RdYlGn', vmin=0.5, vmax=1.0))

### 8.2 File Inventory

In [ ]:
# Create file inventory
file_inventory = {
    'Checkpoints': len(list(CHECKPOINTS_DIR.glob('*.pth'))),
    'JSON Results': len(list(RESULTS_DIR.glob('*.json'))),
    'CSV Results': len(list(RESULTS_DIR.glob('*.csv'))),
    'History Logs': len(list(LOGS_DIR.glob('*.json'))),
    'Training Logs': len(list(BASE_DIR.glob('*.log'))),
    'Figures': len(list(FIGURES_DIR.glob('*.png'))),
    'Grad-CAM Images': len(list((FIGURES_DIR / 'gradcam').glob('*.png'))) if (FIGURES_DIR / 'gradcam').exists() else 0
}

display(Markdown("### Output Files Inventory"))
display_json_as_table(file_inventory)

# Calculate total size
total_size = sum(f.stat().st_size for f in BASE_DIR.rglob('*') if f.is_file())
print(f"\nTotal outputs directory size: {total_size / (1024**3):.2f} GB")

### 8.3 Key Findings & Conclusions

In [ ]:
display(Markdown("""
### Key Findings

#### 1. **Retrained Baseline Performance**
- ✅ Strong internal validation: **84.43% accuracy, 0.9153 QWK**
- ✅ Good generalization on APTOS test: **81.09% accuracy, 0.8959 QWK**
- ⚠️ Significant domain shift on Messidor-2: **62.79% accuracy, 0.6233 QWK**

#### 2. **Domain Shift Analysis**
- **APTOS Val → APTOS Test**: Minimal drop (-3.3% accuracy)
- **APTOS Val → Messidor-2**: Severe drop (-21.6% accuracy, -0.292 QWK)
- **Root cause**: Dataset source difference (Indian vs French hospitals), imaging protocols

#### 3. **Calibration Quality**
- ✅ Well-calibrated on APTOS test: **ECE = 0.048**
- ⚠️ Poorly calibrated on Messidor-2: **ECE = 0.116**
- **Implication**: Confidence estimates less reliable on out-of-distribution data

#### 4. **Class-Specific Challenges**
- **Strong**: No DR detection (F1 > 0.97 on APTOS test)
- **Weak on Messidor-2**: 
  - Mild DR: F1 = 0.109, Recall = 9.3%
  - Moderate DR: Recall = 16.1%
- **Pattern**: Model over-predicts "No DR" on external data

#### 5. **Model Comparison**
- Baseline consistently outperforms CBAM on most metrics
- CBAM attention mechanism did not provide expected benefits
- Retrained baseline serves as solid pre-modification benchmark

#### 6. **Uncertainty Quantification**
- MC Dropout provides useful uncertainty estimates
- Referral curves show accuracy improves when rejecting uncertain cases
- Higher entropy correlates with incorrect predictions

---

### Next Steps
1. ✅ Baseline established with retrained model
2. 🎯 Address domain shift (data augmentation, domain adaptation)
3. 🎯 Improve minority class detection (focal loss, resampling)
4. 🎯 Enhance calibration on external data (temperature scaling)
5. 🎯 Test architectural modifications against this baseline
"""))

---

## End of Report

**Report Generated**: 2026-04-03  
**Total Sections**: 8  
**Models Analyzed**: Baseline (Original, Retrained), CBAM-ResNet50  
**Datasets**: APTOS (train/val/test splits), Messidor-2  

For questions or updates, refer to `docs/result-report.md`

---